# L=2 preliminary Fredenhagen–Marcu (BFFM) phase-transition sweep

Train several **bosonic 3D toric-code** NQS at fixed `hx=0.2` across a grid of
`hz`, then run the `Three_TC.fm` pipeline to plot the electric BFFM order
parameter `O_FM(hz)`, its derivative, and the deduced transition `h_c`.

**This is a mechanical sanity check, not physics.** At `L=2` the FM loop is the
minimal `R=1` and the transition is a broad crossover — the goal is to confirm
the *train → fm_sweep → fit → plot* pipeline runs end-to-end and looks sensible.

> **Prerequisite:** this notebook **clones the GitHub repo** for its code, so
> `Three_TC/fm.py` (the FM pipeline) must be pushed to the remote `main` first.
>
> **Runtime:** set it to **GPU** (Runtime → Change runtime type → GPU) before running.

## 0 · Setup (GPU · clone · install · gate)

In [ ]:
# Confirm a GPU is attached.
!nvidia-smi -L || echo "NO GPU -- switch the runtime to GPU before continuing." 

In [ ]:
# Clone (or update) the repo. Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
REPO_DIR = "/content/ThreeD_TC"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only || true
print("cwd:", os.getcwd())

In [ ]:
# Install the NQS stack, pinned to match the other Colab/NERSC runs.
# (Unrelated dependency-conflict warnings for preinstalled Colab packages are harmless.)
!pip install -q jax==0.5.2 jaxlib==0.5.1 \
    jax-cuda12-plugin==0.5.1 jax-cuda12-pjrt==0.5.1 \
    netket==3.16.1.post1 flax==0.10.4 optax numba wandb tqdm

In [ ]:
# GATE: make sure JAX sees the GPU. Set allocator env BEFORE importing jax.
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
import jax
print("devices:", jax.devices(), "| backend:", jax.default_backend())
assert jax.default_backend() == "gpu", (
    "JAX is not on the GPU -- re-run the install cell, then "
    "Runtime -> Restart session, then re-run from the clone cell.")

## 1 · Configure (the one cell to edit)

In [ ]:
import numpy as np

# ---- physics (fixed point) -------------------------------------------------
L      = 2          # linear size. L=2 -> FM loop is the minimal R=1 (pipeline check).
BC     = "OBC"      # "OBC" (our norm) | "PBC" (the classic L=2 reference)
HX     = 0.2        # transverse field, fixed across the sweep
MODEL  = "bosonic"

# ---- the hz sweep ----------------------------------------------------------
HZ_VALUES = np.round(np.linspace(0.05, 0.95, 10), 3)   # EDIT: grid of hz values

# ---- training hyperparameters ---------------------------------------------
DT         = 2e-2   # (initial) SR learning rate
LR_MIN     = 2e-3   # cosine-decay floor over n_iter (set == DT for a constant lr)
DIAG_SHIFT = 1e-4   # SR / QGT regularisation
N_ITER     = 200
N_SAMPLES  = 8192
N_CHAINS   = 256
QGT        = "dense"  # few params at L=2 -> dense SR is fastest

# ---- architecture (ToricCNN_full): [4,4] / [4,4,1] ------------------------
NONINV = [4, 4]     # pre-Wilson blocks, UNIFORM channels -> noninv_channels, n_noninv
INV    = [4, 4, 1]  # post-Wilson hidden widths AS DISPLAYED (trailing 1 = readout)
assert len(set(NONINV)) == 1, f"noninv channels must be uniform, got {NONINV}"
ARCH_FLAGS = (f"--noninv_channels {NONINV[0]} --n_noninv {len(NONINV)} "
              f"--inv_hidden {' '.join(map(str, INV[:-1]))}")   # model appends the final 1

# ---- output ----------------------------------------------------------------
OUT_DIR = f"outputs/L2_fm_{BC.lower()}_hx{HX}"
os.makedirs(OUT_DIR, exist_ok=True)
print("hz grid :", list(HZ_VALUES))
print("arch    :", ARCH_FLAGS)
print("out_dir :", OUT_DIR)

## 2 · Train the hz sweep

One fresh `python -m Three_TC.train` process per `hz` (isolation = reliability).
Auto-naming `{model}_{arch}_L{L}_hx{hx}_hz{hz}` keeps every run distinct, so all
`*.json` + `*.mpack` land in `OUT_DIR` and the FM sweep can glob them.

In [ ]:
for hz in HZ_VALUES:
    cmd = (f"python -u -m Three_TC.train --L {L} --bc {BC} --model {MODEL} "
           f"--arch ToricCNN_full --hx {HX} --hz {hz} --n_iter {N_ITER} "
           f"--n_samples {N_SAMPLES} --n_chains {N_CHAINS} --qgt {QGT} "
           f"--dt {DT} --lr_min {LR_MIN} --diag_shift {DIAG_SHIFT} "
           f"{ARCH_FLAGS} --out_dir {OUT_DIR} --no_wandb")
    print(f"\n===== training hz={hz} =====")
    !{cmd}

## 3 · Convergence check

The FM order parameter assumes *converged* states. Read each run's JSON and
confirm the V-score is small and the final energy varies smoothly in `hz`.
(Pure JSON read — no diagonalisation.)

In [ ]:
import glob, json
rows = []
for jp in sorted(glob.glob(f"{OUT_DIR}/*.json")):
    d = json.load(open(jp))
    if d.get("config", {}).get("model") != MODEL:
        continue
    o = d.get("observables", {})
    rows.append((d["config"]["hz"], o.get("E0"), o.get("Vscore"),
                 o.get("sz_mean"), d.get("runtime_s")))
rows.sort()
print(f"{'hz':>6} {'E0':>12} {'Vscore':>11} {'<sz>':>9} {'runtime_s':>10}")
for hz, e0, vsc, sz, rt in rows:
    print(f"{hz:6.3f} {e0:12.5f} {vsc:11.3e} {sz:9.4f} {rt:10.1f}")
print(f"\n{len(rows)} runs in {OUT_DIR}. Vscore small + smooth energies => trust the FM.")

## 4 · FM order parameter, derivative, and h_c

`fm_sweep` reloads each trained NQS, evaluates the electric half-square BFFM
ratio `O_FM = ⟨open⟩/√|⟨closed⟩|` (plus ⟨σz⟩), and `fit_transition`
locates `h_c` as the logistic inflection (= derivative peak), with a
finite-difference peak as a model-free cross-check.

In [ ]:
import importlib
from Three_TC import fm
importlib.reload(fm)          # pick up the freshly-pulled pipeline code

tab = fm.fm_sweep(OUT_DIR, sector="electric", field="hz",
                  L=L, hx=HX, model=MODEL, bc=BC, eval_samples=16384)
fit = fm.fit_transition(tab["field"], tab["O"], tab["Oe"])
print(f"\nh_c(L={L}) = {fit['h_c']:.4f} ± {fit['h_c_err']:.4f}   |   FD peak = {fit['h_c_fd']:.4f}")

import matplotlib.pyplot as plt
fm.plot_fm_sweep(tab["field"], tab["O"], tab["Oe"], fit, sector="electric", L=L)
plt.show()

In [ ]:
# Diagonal cross-check: <sigma_z> and its susceptibility should peak near the FM h_c.
f, mz = tab["field"], tab["mz"]
dmz = np.gradient(mz, f)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].errorbar(f, mz, yerr=tab["mz_e"], fmt="o-", capsize=3)
ax[0].set(xlabel="hz", ylabel=r"$\langle \sigma_z \rangle$", title="diagonal cross-check")
ax[1].plot(f, dmz, "s-")
ax[1].axvline(fit["h_c"], ls="--", c="k", label=f"FM $h_c$={fit['h_c']:.3f}")
ax[1].set(xlabel="hz", ylabel=r"$d\langle\sigma_z\rangle/dh_z$", title="susceptibility")
ax[1].legend(); plt.tight_layout(); plt.show()

## 5 · (optional) download the trained checkpoints

In [ ]:
# from google.colab import files
# !zip -qr /content/L2_fm_runs.zip {OUT_DIR} && echo "zipped {OUT_DIR}"
# files.download("/content/L2_fm_runs.zip")